In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
from pathlib import Path

split_dir = Path("/content/drive/MyDrive/NLP/splits")

train_df = pd.read_csv(split_dir / "train.csv", keep_default_na=False)
val_df = pd.read_csv(split_dir / "validation.csv", keep_default_na=False)
test_df = pd.read_csv(split_dir / "test.csv", keep_default_na=False)

X_train = train_df["review_comment"].to_numpy(dtype=str)
X_val = val_df["review_comment"].to_numpy(dtype=str)
X_test = test_df["review_comment"].to_numpy(dtype=str)

y_train = train_df["label"].to_numpy(dtype="int32")
y_val = val_df["label"].to_numpy(dtype="int32")
y_test = test_df["label"].to_numpy(dtype="int32")

In [3]:
import tensorflow as tf
from tensorflow.keras import layers, models

tf.keras.utils.set_random_seed(42)

BATCH_SIZE = 64
MAX_TOKENS = 15000

def make_dataset(texts, labels, training=False):
    ds = tf.data.Dataset.from_tensor_slices((texts, labels))

    if training:
        ds = ds.shuffle(
            buffer_size=len(texts),
            seed=42
        )

    return ds.batch(BATCH_SIZE).prefetch(1)

In [4]:
train_ds = make_dataset(X_train, y_train, training=True)
val_ds = make_dataset(X_val, y_val)
test_ds = make_dataset(X_test, y_test)

In [5]:
vectorizer = layers.TextVectorization(
    max_tokens=MAX_TOKENS,
    output_mode="int",
    output_sequence_length=150
)

adapt_ds = tf.data.Dataset.from_tensor_slices(X_train).batch(BATCH_SIZE)
vectorizer.adapt(adapt_ds)

vocabulary = vectorizer.get_vocabulary()

word_to_index = {
    word: index for index, word in enumerate(vocabulary)
}

print("Vocabulary size:", len(vocabulary))

Vocabulary size: 15000


In [6]:
from pathlib import Path
import urllib.request

embedding_dir = Path("/content/drive/MyDrive/NLP/embeddings")
embedding_dir.mkdir(parents=True, exist_ok=True)

vectors_path = embedding_dir / "cc.pt.300.vec.gz"

url = (
    "https://dl.fbaipublicfiles.com/"
    "fasttext/vectors-crawl/cc.pt.300.vec.gz"
)

if not vectors_path.exists():
    temporary_path = vectors_path.with_suffix(".download")
    urllib.request.urlretrieve(url, temporary_path)
    temporary_path.replace(vectors_path)

print("Vectors available:", vectors_path)

Vectors available: /content/drive/MyDrive/NLP/embeddings/cc.pt.300.vec.gz


In [7]:
import gzip
import numpy as np

EMBEDDING_DIM = 300

# Unmatched words receive small random vectors.
rng = np.random.default_rng(42)

embedding_matrix = rng.normal(
    loc=0.0,
    scale=0.05,
    size=(len(vocabulary), EMBEDDING_DIM)
).astype("float32")

# Token 0 is padding
embedding_matrix[0] = 0.0

# Exclude padding and the shared unknown-word token
needed_words = set(vocabulary[2:])
matched_words = set()

with gzip.open(vectors_path, "rt", encoding="utf-8") as stream:
    header = stream.readline().split()
    vector_dimension = int(header[1])

    if vector_dimension != EMBEDDING_DIM:
        raise ValueError(f"Unexpected vector dimension: {vector_dimension}")

    for line in stream:
        word, separator, values = line.rstrip().partition(" ")

        if not separator or word not in needed_words:
            continue

        vector = np.fromstring(values, sep=" ", dtype="float32")

        if vector.shape != (EMBEDDING_DIM,):
            continue

        embedding_matrix[word_to_index[word]] = vector
        matched_words.add(word)

        if len(matched_words) == len(needed_words):
            break

coverage = len(matched_words) / max(len(needed_words), 1)

print(f"Matched words: {len(matched_words):,}/{len(needed_words):,}")
print(f"Vocabulary coverage: {coverage:.1%}")
print(f"Embedding matrix RAM: {embedding_matrix.nbytes / 1024**2:.1f} MB")

Matched words: 11,620/14,998
Vocabulary coverage: 77.5%
Embedding matrix RAM: 17.2 MB


In [8]:
model = models.Sequential([
    layers.Input(shape=(), dtype=tf.string),

    vectorizer,

    layers.Embedding(
        input_dim=len(vocabulary),
        output_dim=EMBEDDING_DIM,
        embeddings_initializer=tf.keras.initializers.Constant(
            embedding_matrix
        ),
        trainable=False,
        name="fasttext_embedding"
    ),

    layers.Conv1D(
        filters=128,
        kernel_size=3,
        activation="relu"
    ),

    layers.GlobalMaxPooling1D(),
    layers.Dropout(0.3),
    layers.Dense(5, activation="softmax")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization              │ (None, 150)            │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fasttext_embedding (Embedding)  │ (None, 150, 300)       │     4,500,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 148, 128)       │       115,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d            │ (None, 128)            │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 5)              │           645 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,615,973 (17.61 MB)

 Trainable params: 115,973 (453.02 KB)

 Non-trainable params: 4,500,000 (17.17 MB)

In [9]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.unique(y_train)

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weights = {
    int(label): float(weight)
    for label, weight in zip(classes, weights)
}

print(class_weights)

{0: 0.9406764333514198, 1: 3.8355457227138645, 2: 2.3033658104517274, 3: 1.3694049499736705, 4: 0.39811696264543783}


In [10]:
gentle_weights = {
    label: float(np.sqrt(weight))
    for label, weight in class_weights.items()
}

print(gentle_weights)

{0: 0.9698847526131236, 1: 1.958454932520497, 2: 1.5176843579782087, 3: 1.1702157706908887, 4: 0.6309651041424065}


In [11]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    class_weight=gentle_weights,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=3,
            restore_best_weights=True
        )
    ]
)

Epoch 1/50
407/407 ━━━━━━━━━━━━━━━━━━━━ 8s 8ms/step - accuracy: 0.6265 - loss: 1.1419 - val_accuracy: 0.6692 - val_loss: 0.9487
Epoch 2/50
407/407 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.6652 - loss: 1.0278 - val_accuracy: 0.6787 - val_loss: 0.9091
Epoch 3/50
407/407 ━━━━━━━━━━━━━━━━━━━━ 7s 11ms/step - accuracy: 0.6797 - loss: 0.9858 - val_accuracy: 0.6783 - val_loss: 0.9031
Epoch 4/50
407/407 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.6854 - loss: 0.9601 - val_accuracy: 0.6879 - val_loss: 0.8742
Epoch 5/50
407/407 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.6919 - loss: 0.9337 - val_accuracy: 0.6783 - val_loss: 0.8922
Epoch 6/50
407/407 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.6957 - loss: 0.9156 - val_accuracy: 0.6819 - val_loss: 0.8776
Epoch 7/50
407/407 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - accuracy: 0.7023 - loss: 0.8974 - val_accuracy: 0.6912 - val_loss: 0.8581
Epoch 8/50
407/407 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.7091 - loss: 0.8762 - val_accuracy: 

In [12]:
import numpy as np
from sklearn.metrics import classification_report, mean_absolute_error

# Validation predictions while developing the model
y_pred = np.argmax(model.predict(val_ds), axis=1)

print(classification_report(
    y_val,
    y_pred,
    labels=[0, 1, 2, 3, 4],
    target_names=["Score 1", "Score 2", "Score 3", "Score 4", "Score 5"],
    zero_division=0
))

print("MAE:", mean_absolute_error(y_val, y_pred))

102/102 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step
              precision    recall  f1-score   support

     Score 1       0.65      0.90      0.75      1383
     Score 2       0.17      0.05      0.08       339
     Score 3       0.28      0.18      0.22       564
     Score 4       0.39      0.16      0.22       950
     Score 5       0.80      0.91      0.85      3266

    accuracy                           0.69      6502
   macro avg       0.46      0.44      0.43      6502
weighted avg       0.63      0.69      0.64      6502

MAE: 0.4843125192248539


In [13]:
import numpy as np
import pandas as pd

errors = pd.DataFrame({
    "comment": np.asarray(X_val).ravel(),
    "actual": np.asarray(y_val).ravel() + 1,
    "predicted": np.asarray(y_pred).ravel() + 1
})

confusion_pct = pd.crosstab(
    errors["actual"], errors["predicted"], normalize="index"
).reindex(index=range(1, 6), columns=range(1, 6), fill_value=0)

display((100 * confusion_pct).round(1))

missed_twos = errors.query("actual == 2 and predicted != 2")
with pd.option_context("display.max_colwidth", None):
    display(missed_twos.sample(min(12, len(missed_twos)), random_state=42))

predicted,1,2,3,4,5
actual,,,,,
1,90.2,2.6,4.8,0.5,1.9
2,71.7,5.3,12.4,1.8,8.8
3,43.1,5.1,18.4,14.0,19.3
4,10.7,1.9,11.4,15.7,60.3
5,2.9,0.1,1.7,4.3,91.1


,comment,actual,predicted
3333,"Não entendo o que está acontecendo, me parecia uma compra e um produto bem simples.",2,1
2583,"boa tarde só me foi entregue um relógio falta o de perola que não entregaram, gostaria de saber quando vão entregar o outro.",2,1
3897,não recebi o produto consta que o mesmo voltou para o remetente. como faço para que possa reviar-lo?,2,1
294,"Recebi dois carregadores, porem com referencias diferentes. Modelo CDQ-X28 (CORRETO), e o outro que dei de presente e NAO SERVIU.",2,1
2193,"O perfume que recebi tem embalagem diferente da foto do site e também diferente de um perfume igual que tenho em casa. O cheiro é bem parecido, mas não igual. Enfim, parece falsificado.",2,1
2392,"Produto de boa qualidade, porém, comprei para o modelo errado do g5.",2,3
4978,Já havia até desistido de esperar quando o produto chegou,2,1
3102,muito bom.,2,5
4376,Sempre recebi no prazo mais ainda não recebi,2,1
6191,produto horrivel produto fora da minha expectativa,2,1


In [14]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error

probs = model.predict(val_ds, verbose=0)
actual = np.asarray(y_val).ravel() + 1

predictions = {
    "Most probable star (current)": probs.argmax(axis=1) + 1,
    "Median star": (probs.cumsum(axis=1) < 0.5).sum(axis=1) + 1,
}

rows = []
for rule, pred in predictions.items():
    rows.append({
        "Rule": rule,
        "Accuracy": accuracy_score(actual, pred),
        "Macro-F1": f1_score(actual, pred, labels=[1, 2, 3, 4, 5],
                             average="macro", zero_division=0),
        "MAE": mean_absolute_error(actual, pred),
        "Two-star recall": (pred[actual == 2] == 2).mean(),
        "Within one star": (np.abs(actual - pred) <= 1).mean(),
    })

display(pd.DataFrame(rows).set_index("Rule").round(4))

,Accuracy,Macro-F1,MAE,Two-star recall,Within one star
Rule,,,,,
Most probable star (current),0.6912,0.4266,0.4843,0.0531,0.8834
Median star,0.6349,0.4557,0.4918,0.2389,0.9031


In [15]:
# from pathlib import Path

# model_dir = Path("/content/drive/MyDrive/NLP/models")
# model_dir.mkdir(parents=True, exist_ok=True)

# model_path = model_dir / "review_score_model_3.keras"

# model.save(str(model_path))

# print("Model saved:", model_path)